# iclr26-2-freeze-n12

Freezes the 12-atom census for the ICLR quantum-circuit study.

Inputs: `qm9_panel_columns.pkl` from `iclr26-1-dataset-construction`, and QM9 through PyTorch Geometric for the bond graphs.

What it fixes:
- the census: every QM9 molecule with exactly 12 atoms (hydrogens included), valence-error records excluded
- per-molecule graphs: atomic numbers, typed edge list, bond lengths, gated against the panel
- census constants the producer normalizes by: maximum degree, longest bond
- the probe population (one representative per connectivity class) and the gate set (duplicates and stereo twins, labeled)
- five folds with every formula bucket kept whole
- the retained targets, the excluded ones with reasons, and the degenerate-fold flags
- synchronized sector vocabularies for the element, degree, and joint quotients
- digests of everything above

Output: one pickle, `iclr26_freeze_n12.pkl`, carrying the config that produced it.

In [22]:
!pip install torch_geometric rdkit

In [23]:
import time
import pickle
import hashlib
import itertools
import numpy as np
import networkx as nx
from networkx.algorithms import isomorphism
from rdkit import Chem, RDLogger
from torch_geometric.datasets import QM9
RDLogger.DisableLog("rdApp.*")

In [24]:
# ---------------- config: every constant, seed, and path lives here ----------------
PANEL_PKL = "/kaggle/input/notebooks/lukemiller1987/iclr26-1-dataset-construction/qm9_panel_columns.pkl"       # written by iclr26-1-dataset-construction; on Kaggle point this at the input-dataset copy
QM9_ROOT = "qm9"                           # PyG download and processing root, same as the dataset notebook
OUTPUT_PKL = "iclr26_freeze_n12.pkl"

CENSUS_NUM_ATOMS = 12                      # exactly this many atoms, hydrogens included
EXPECTED_MAX_DEGREE = 4                    # census-wide maximum degree; the producer normalizes degree angles by it
NUM_FOLDS = 5
SEED = 0                                   # breaks ties between equal-size formula buckets in the fold assignment
MIN_DISTINCT_PER_TEST_FOLD = 2             # a target is flagged in every fold whose test molecules show fewer distinct values; nothing is excluded
PROGRESS_EVERY = 500                       # molecules between elapsed-time prints

ELEMENTS = [1, 6, 7, 8, 9]                 # H C N O F, the order of the panel's formula tuples
ELEMENT_SYMBOLS = {1: "H", 6: "C", 7: "N", 8: "O", 9: "F"}
PYG_BOND_TYPES = ["single", "double", "triple", "aromatic"]      # column order of PyG's QM9 edge_attr one-hot (Kekule types: PyG reads the SDF unsanitized)
RDKIT_BOND_TYPE_NAMES = {Chem.BondType.SINGLE: "single", Chem.BondType.DOUBLE: "double",
                         Chem.BondType.TRIPLE: "triple", Chem.BondType.AROMATIC: "aromatic"}
BOND_ORDER_OF_TYPE = {"single": 1, "double": 2, "triple": 3}                     # for the sanitization-cleanup check only
DEGREE_CLASSES = list(range(1, EXPECTED_MAX_DEGREE + 1))         # degree classes for the degree-sector quotient

# Target blocks by panel section. Every scalar panel field must appear in exactly one block or in NON_TARGET_FIELDS.
TARGET_BLOCKS = {
    "composition": [
        "num_atoms", "num_heavy_atoms", "num_bonds_full", "num_bonds_heavy",
        "num_single_bonds_heavy", "num_double_bonds_heavy", "num_triple_bonds_heavy", "num_aromatic_bonds_heavy"],
    "rings": [
        "num_rings", "num_aromatic_rings", "num_nonaromatic_rings",
        "num_3_member_rings", "num_4_member_rings", "num_5_member_rings", "num_6_member_rings", "num_7_plus_member_rings",
        "num_aromatic_5_member_rings", "num_aromatic_6_member_rings", "num_benzene_rings",
        "num_heteroaromatic_5_member_rings", "num_heteroaromatic_6_member_rings"],
    "ring_systems": [
        "num_benzene_isolated", "num_benzene_fused_terminal", "num_benzene_fused_interior", "num_benzene_fused_branch",
        "num_aromatic_isolated", "num_aromatic_fused_terminal", "num_aromatic_fused_interior", "num_aromatic_fused_branch",
        "num_aromatic_spiro", "num_ring_systems", "largest_ring_system_size", "num_fused_ring_systems",
        "num_spiro_centers", "max_ring_system_degree"],
    "substitution": [
        "num_aromatic_monosubstituted", "num_aromatic_disubstituted", "num_aromatic_trisubstituted_plus",
        "num_ortho_pairs", "num_meta_pairs", "num_para_pairs"],
    "conjugation": [
        "num_conjugated_bonds", "num_conjugated_atoms", "num_conjugated_components", "largest_conjugated_component_atoms",
        "longest_conjugated_path", "num_aromatic_atoms", "aromatic_atom_fraction", "num_heteroatoms_in_conjugated_systems",
        "num_ring_conjugated_components"],
    "invariants": [
        "num_bridges", "num_articulation_points", "cyclomatic_number", "girth",
        "diameter_heavy", "radius_heavy", "wiener_index_heavy", "average_shortest_path_length_heavy",
        "adjacency_spectral_radius_heavy", "laplacian_algebraic_connectivity_heavy",
        "diameter_full", "radius_full", "wiener_index_full", "adjacency_spectral_radius_full", "laplacian_algebraic_connectivity_full",
        "triangle_count", "cycle_4_count", "cycle_5_count", "cycle_6_count", "num_stereocenters"],
    "geometry": [
        "radius_of_gyration_full", "radius_of_gyration_heavy",
        "heavy_pairwise_distance_mean", "heavy_pairwise_distance_std", "heavy_pairwise_distance_min", "heavy_pairwise_distance_max",
        "bond_length_mean", "bond_length_std", "bond_length_min", "bond_length_max",
        "num_heavy_angles", "heavy_angle_mean", "heavy_angle_std", "heavy_angle_min", "heavy_angle_max",
        "num_heavy_torsions", "num_rotatable_bonds", "torsion_abs_cos_mean_rotatable", "torsion_abs_cos_mean_nonrotatable"],
    "functional_groups": [
        "num_hydroxyl", "num_phenol", "num_ether", "num_carbonyl", "num_aldehyde", "num_ketone", "num_carboxylic_acid",
        "num_ester", "num_amide", "num_amine_primary", "num_amine_secondary", "num_amine_tertiary", "num_nitrile",
        "num_nitro", "num_alkene", "num_alkyne", "num_fluoro"],
    "qm9_properties": [
        "mu", "alpha", "homo", "lumo", "gap", "r2", "zpve", "U0", "U", "H", "G", "Cv",
        "U0_atom", "U_atom", "H_atom", "G_atom", "A", "B", "C"],
}
NON_TARGET_FIELDS = [
    "valence_error", "formula_class_id", "connectivity_class_id", "num_unassigned_stereocenters",
    "num_duplicates", "num_constitutional_isomers", "num_enantiomers", "num_diastereomers",
    "has_constitutional_isomer", "has_enantiomer", "has_diastereomer"]
TARGET_NAMES = [name for block in TARGET_BLOCKS.values() for name in block]

## Panel and census selection

In [25]:
with open(PANEL_PKL, "rb") as handle:
    panel = pickle.load(handle)
panel_config = panel["config"]
columns = panel["columns"]
num_panel_molecules = panel_config["num_molecules"]
assert len(columns["num_atoms"]) == num_panel_molecules, (len(columns["num_atoms"]), num_panel_molecules)

# every scalar field is either a target or explicitly not one; nothing is dropped by omission
panel_scalar_fields = list(panel_config["PANEL_SCALAR_FIELDS"])
accounted_fields = TARGET_NAMES + NON_TARGET_FIELDS
assert len(accounted_fields) == len(set(accounted_fields)), "a field is listed twice in TARGET_BLOCKS / NON_TARGET_FIELDS"
missing_here = sorted(set(panel_scalar_fields) - set(accounted_fields))
unknown_here = sorted(set(accounted_fields) - set(panel_scalar_fields))
assert not missing_here and not unknown_here, f"panel fields not accounted for: {missing_here}; listed but absent from panel: {unknown_here}"
assert TARGET_BLOCKS["functional_groups"] == list(panel_config["FUNCTIONAL_GROUP_SMARTS"].keys())
assert TARGET_BLOCKS["qm9_properties"] == list(panel_config["QM9_TARGET_NAMES"])

at_census_size = columns["num_atoms"] == CENSUS_NUM_ATOMS
valence_error = columns["valence_error"].astype(bool)
census_rows = np.where(at_census_size & ~valence_error)[0]          # row = position in the panel columns = position in the PyG dataset
print(f"panel molecules: {num_panel_molecules}")
print(f"molecules with {CENSUS_NUM_ATOMS} atoms: {int(at_census_size.sum())}")
print(f"  valence-error records excluded: {int((at_census_size & valence_error).sum())}")
print(f"  census: {len(census_rows)}")
assert len(census_rows) > 0

panel molecules: 130831
molecules with 12 atoms: 2189
  valence-error records excluded: 53
  census: 2136


## Bond graphs from PyG, gated against the panel

Row position in the panel columns equals position in the PyG dataset (the dataset notebook built molecules in dataset order), and the three identity asserts below check that on every census molecule.

Two bond-type vocabularies exist for the same edges. PyG reads the SDF with `sanitize=False`, so `edge_attr` carries the SDF's Kekulé types (alternating single and double around a benzene ring, never aromatic). The panel parsed the SMILES with sanitization on, so RDKit perceived aromaticity there, and every ring target in the panel uses that perception. The circuits use the panel's convention, because the settled bond-type table gives aromatic bonds order 1.5, and because a Kekulé assignment is an arbitrary choice the SDF made, which a permutation-invariant readout should not inherit. So each molecule gets `bond_types` (perceived, panel convention) and `kekule_bond_types` (PyG's). The perceived types are read by parsing QM9's SMILES unsanitized, aligning it to PyG's atom order with a match on elements and Kekulé bond types, then sanitizing; the gates check the perceived counts against the panel and the Kekulé types against PyG edge by edge. Sanitization changes a Kekulé type in exactly two ways: aromaticity perception, and RDKit's charge-separation cleanup of hypervalent nitrogen groups that the SDF writes neutral (nitro N(=O)=O becomes [N+](=O)[O-], azide N=N#N becomes N=[N+]=[N-]), which lowers one bond order by one and charges its endpoints. The panel took the same route, so its bond types and functional-group counts already reflect the cleanup; the per-edge check allows exactly those two changes and nothing else.

In [26]:
start = time.time()
dataset = QM9(root=QM9_ROOT)
print(f"QM9 loaded: {len(dataset)} molecules, {time.time() - start:.0f}s")
assert len(dataset) == num_panel_molecules, (len(dataset), num_panel_molecules)


def graph_record_from_pyg(data, row):
    """Atomic numbers, sorted undirected edge list with bond types, and bond lengths for one PyG QM9 record"""
    atomic_numbers = data.z.numpy().astype(int)
    positions = data.pos.numpy().astype(float)
    edge_index = data.edge_index.numpy()
    edge_attr = data.edge_attr.numpy()
    assert edge_attr.shape[1] == len(PYG_BOND_TYPES), edge_attr.shape
    assert np.all(edge_attr.sum(axis=1) == 1), f"row {row}: edge_attr is not one-hot"
    bond_type_by_pair = {}
    for (atom_a, atom_b), one_hot in zip(edge_index.T, edge_attr):
        pair = (int(min(atom_a, atom_b)), int(max(atom_a, atom_b)))
        bond_type = PYG_BOND_TYPES[int(np.argmax(one_hot))]
        assert bond_type_by_pair.get(pair, bond_type) == bond_type, f"row {row}: bond type differs between the two directions of {pair}"
        bond_type_by_pair[pair] = bond_type
    assert 2 * len(bond_type_by_pair) == edge_index.shape[1], f"row {row}: an edge is missing its reverse direction"
    edges = np.array(sorted(bond_type_by_pair), dtype=int).reshape(-1, 2)
    kekule_bond_types = [bond_type_by_pair[(int(edge[0]), int(edge[1]))] for edge in edges]
    bond_lengths = np.array([np.linalg.norm(positions[edge[0]] - positions[edge[1]]) for edge in edges])
    return atomic_numbers, edges, kekule_bond_types, bond_lengths


def perceived_bond_types(smiles, atomic_numbers, edges, kekule_bond_types, row):
    """Panel-convention bond types (RDKit aromaticity perception) in PyG edge order: parse the smiles unsanitized, align to PyG order on elements and Kekule types, sanitize, read"""
    parser_params = Chem.SmilesParserParams()
    parser_params.removeHs = False
    parser_params.sanitize = False
    mol = Chem.MolFromSmiles(smiles, parser_params)
    assert mol is not None and mol.GetNumAtoms() == len(atomic_numbers), f"row {row}: smiles parse"

    pyg_graph = nx.Graph()
    pyg_graph.add_nodes_from((int(index), {"element": int(element)}) for index, element in enumerate(atomic_numbers))
    pyg_graph.add_edges_from((int(edge[0]), int(edge[1]), {"bond_type": bond_type}) for edge, bond_type in zip(edges, kekule_bond_types))
    rdkit_graph = nx.Graph()
    rdkit_graph.add_nodes_from((atom.GetIdx(), {"element": atom.GetAtomicNum()}) for atom in mol.GetAtoms())
    rdkit_graph.add_edges_from((bond.GetBeginAtomIdx(), bond.GetEndAtomIdx(), {"bond_type": RDKIT_BOND_TYPE_NAMES[bond.GetBondType()]}) for bond in mol.GetBonds())
    matcher = isomorphism.GraphMatcher(pyg_graph, rdkit_graph,
                                       node_match=isomorphism.categorical_node_match("element", None),
                                       edge_match=isomorphism.categorical_edge_match("bond_type", None))
    assert matcher.is_isomorphic(), f"row {row}: PyG typed graph and smiles typed graph are not isomorphic"
    pyg_to_rdkit = matcher.mapping
    mol = Chem.RenumberAtoms(mol, [pyg_to_rdkit[pyg_index] for pyg_index in range(len(atomic_numbers))])
    sanitize_result = Chem.SanitizeMol(mol, catchErrors=True)
    assert sanitize_result == Chem.SanitizeFlags.SANITIZE_NONE, f"row {row}: sanitization failed with {sanitize_result}"

    perceived_by_pair = {}
    for bond in mol.GetBonds():
        pair = (min(bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()), max(bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()))
        perceived_by_pair[pair] = RDKIT_BOND_TYPE_NAMES[bond.GetBondType()]
    assert set(perceived_by_pair) == {(int(edge[0]), int(edge[1])) for edge in edges}, f"row {row}: edge sets differ after renumbering"
    bond_types = [perceived_by_pair[(int(edge[0]), int(edge[1]))] for edge in edges]
    formal_charges = np.array([atom.GetFormalCharge() for atom in mol.GetAtoms()], dtype=int)

    # sanitization may change a Kekule type in exactly two ways: aromaticity perception, or the charge-separation cleanup
    # RDKit applies to hypervalent nitrogen groups written neutral in the SDF (nitro N(=O)=O -> [N+](=O)[O-], azide N=N#N -> N=[N+]=[N-]),
    # which lowers one bond order by one and puts charges +1 and -1 on its endpoints. Anything else is an error.
    charge_separated_bonds = []
    for edge, perceived, kekule in zip(edges, bond_types, kekule_bond_types):
        if perceived == kekule:
            continue
        if perceived == "aromatic":
            assert kekule in ("single", "double"), f"row {row}: aromatic bond over Kekule type {kekule}"
            continue
        endpoint_charges = sorted((int(formal_charges[edge[0]]), int(formal_charges[edge[1]])))
        order_drop = BOND_ORDER_OF_TYPE[kekule] - BOND_ORDER_OF_TYPE[perceived]
        assert order_drop == 1 and endpoint_charges == [-1, 1],             f"row {row}: unexplained bond change {kekule} -> {perceived} on {ELEMENT_SYMBOLS[int(atomic_numbers[edge[0]])]}-{ELEMENT_SYMBOLS[int(atomic_numbers[edge[1]])]} with charges {endpoint_charges}"
        charge_separated_bonds.append((int(edge[0]), int(edge[1])))
    return bond_types, formal_charges, charge_separated_bonds


molecules = []
start = time.time()
for count, row in enumerate(census_rows):
    row = int(row)
    data = dataset[row]
    assert int(data.idx) == int(columns["qm9_index"][row]), f"row {row}: qm9_index mismatch"
    assert data.name == columns["name"][row], f"row {row}: name mismatch"
    assert data.smiles == columns["smiles"][row], f"row {row}: smiles mismatch"
    atomic_numbers, edges, kekule_bond_types, bond_lengths = graph_record_from_pyg(data, row)
    assert len(atomic_numbers) == CENSUS_NUM_ATOMS, f"row {row}: {len(atomic_numbers)} atoms"
    bond_types, formal_charges, charge_separated_bonds = perceived_bond_types(data.smiles, atomic_numbers, edges, kekule_bond_types, row)

    # gates against the panel: composition, edge count, degree histogram, bond lengths, perceived bond types (heavy counts and full counts)
    formula = tuple(int((atomic_numbers == element).sum()) for element in ELEMENTS)
    assert formula == tuple(columns["formula"][row]), f"row {row}: formula {formula} vs panel {columns['formula'][row]}"
    assert len(edges) == int(columns["num_bonds_full"][row]), f"row {row}: edge count"
    degrees = np.bincount(edges.ravel(), minlength=CENSUS_NUM_ATOMS)
    degree_histogram = tuple(int((degrees == degree).sum()) for degree in range(6))
    assert degree_histogram == tuple(columns["degree_histogram_full"][row]), f"row {row}: degree histogram"
    assert np.allclose(np.sort(bond_lengths), np.array(columns["bond_length_spectrum"][row]), atol=1e-9), f"row {row}: bond lengths"
    heavy_edge = (atomic_numbers[edges[:, 0]] > 1) & (atomic_numbers[edges[:, 1]] > 1)
    for bond_type in PYG_BOND_TYPES:
        heavy_count = sum(1 for is_heavy, this_type in zip(heavy_edge, bond_types) if is_heavy and this_type == bond_type)
        assert heavy_count == int(columns[f"num_{bond_type}_bonds_heavy"][row]), f"row {row}: heavy {bond_type} count"
        full_count = sum(len(lengths) for key, lengths in columns["bond_lengths_by_type"][row].items() if key[2] == bond_type)
        assert full_count == bond_types.count(bond_type), f"row {row}: full {bond_type} count"
    graph = nx.Graph()
    graph.add_nodes_from(range(CENSUS_NUM_ATOMS))
    graph.add_edges_from((int(edge[0]), int(edge[1])) for edge in edges)
    assert nx.is_connected(graph), f"row {row}: disconnected bond graph"

    molecules.append({
        "row": row,
        "qm9_index": int(columns["qm9_index"][row]),
        "name": columns["name"][row],
        "smiles": columns["smiles"][row],
        "canonical_smiles": columns["canonical_smiles"][row],
        "flat_smiles": columns["flat_smiles"][row],
        "formula": formula,
        "atomic_numbers": atomic_numbers,
        "degrees": degrees,
        "edges": edges,
        "bond_types": bond_types,                    # RDKit-perceived, the panel's convention; the bond-type arm uses these
        "kekule_bond_types": kekule_bond_types,      # PyG edge_attr, the SDF's Kekule assignment; kept for provenance
        "formal_charges": formal_charges,            # after RDKit sanitization; nonzero only in charge-separated groups
        "charge_separated_bonds": charge_separated_bonds,   # edges whose Kekule order the cleanup lowered by one
        "bond_lengths": bond_lengths,
        "connectivity_class_id": int(columns["connectivity_class_id"][row]),
        "num_unassigned_stereocenters": int(columns["num_unassigned_stereocenters"][row]),
    })
    if (count + 1) % PROGRESS_EVERY == 0:
        print(f"  {count + 1} of {len(census_rows)} molecules, {time.time() - start:.0f}s elapsed")
print(f"{len(molecules)} molecules built and gated in {time.time() - start:.0f}s")
aromatic_molecules = sum(any(bond_type == "aromatic" for bond_type in molecule["bond_types"]) for molecule in molecules)
kekule_aromatic = sum(any(bond_type == "aromatic" for bond_type in molecule["kekule_bond_types"]) for molecule in molecules)
print(f"molecules with perceived aromatic bonds: {aromatic_molecules}; with Kekule-typed aromatic bonds from PyG: {kekule_aromatic} (expected 0)")
charge_separated_molecules = sum(len(molecule["charge_separated_bonds"]) > 0 for molecule in molecules)
print(f"molecules with a charge-separated group after sanitization (nitro, azide): {charge_separated_molecules}")

# census constants the producer normalizes by
census_max_degree = max(int(molecule["degrees"].max()) for molecule in molecules)
assert census_max_degree == EXPECTED_MAX_DEGREE, f"census maximum degree is {census_max_degree}, config expects {EXPECTED_MAX_DEGREE}"
census_longest_bond = max(float(molecule["bond_lengths"].max()) for molecule in molecules)
census_shortest_bond = min(float(molecule["bond_lengths"].min()) for molecule in molecules)
print(f"census maximum degree {census_max_degree}; bond lengths {census_shortest_bond:.4f} to {census_longest_bond:.4f} angstrom")

QM9 loaded: 130831 molecules, 2s
  500 of 2136 molecules, 1s elapsed
  1000 of 2136 molecules, 5s elapsed
  1500 of 2136 molecules, 6s elapsed
  2000 of 2136 molecules, 7s elapsed
2136 molecules built and gated in 8s
molecules with perceived aromatic bonds: 1328; with Kekule-typed aromatic bonds from PyG: 0 (expected 0)
molecules with a charge-separated group after sanitization (nitro, azide): 29
census maximum degree 4; bond lengths 0.9606 to 1.7567 angstrom


## Probe population and gate set

Connectivity classes are the panel's (same flat SMILES, hence the same typed bond graph). The representative is the lowest row in each class; the rest form the gate set. Pair relations are recomputed with the panel's own rule and asserted against the panel's per-molecule counts wherever the class lost no members to the valence-error exclusion.

In [27]:
def mirror_smiles_of(canonical_smiles):
    """Canonical smiles of the mirror image: every tetrahedral tag inverted, double-bond stereo untouched (the panel's convention)"""
    heavy_mol = Chem.MolFromSmiles(canonical_smiles)
    assert heavy_mol is not None, f"RDKit could not parse {canonical_smiles}"
    mirror = Chem.RWMol(heavy_mol)
    for atom in mirror.GetAtoms():
        if atom.GetChiralTag() == Chem.ChiralType.CHI_TETRAHEDRAL_CW:
            atom.SetChiralTag(Chem.ChiralType.CHI_TETRAHEDRAL_CCW)
        elif atom.GetChiralTag() == Chem.ChiralType.CHI_TETRAHEDRAL_CCW:
            atom.SetChiralTag(Chem.ChiralType.CHI_TETRAHEDRAL_CW)
    return Chem.MolToSmiles(mirror.GetMol())


def classify_pair(molecule_a, molecule_b):
    """Two molecules with the same flat smiles: duplicate, enantiomer, diastereomer, or stereo_undefined (the panel's rule, same order of tests)"""
    if molecule_a["canonical_smiles"] == molecule_b["canonical_smiles"]:
        return "duplicate"
    if molecule_a["num_unassigned_stereocenters"] > 0 or molecule_b["num_unassigned_stereocenters"] > 0:
        return "stereo_undefined"
    if molecule_a["canonical_smiles"] == mirror_smiles_of(molecule_b["canonical_smiles"]):
        return "enantiomer"
    return "diastereomer"


# classes, in ascending row order within each class
classes = {}
for position, molecule in enumerate(molecules):
    classes.setdefault(molecule["connectivity_class_id"], []).append(position)
for class_id, positions in classes.items():
    flat_smiles_seen = {molecules[position]["flat_smiles"] for position in positions}
    formulas_seen = {molecules[position]["formula"] for position in positions}
    assert len(flat_smiles_seen) == 1 and len(formulas_seen) == 1, f"class {class_id} mixes flat smiles or formulas"

# panel-wide class sizes, to know which classes lost members to the valence-error exclusion
panel_class_sizes = {}
for class_id in np.unique(columns["connectivity_class_id"][at_census_size]):
    panel_class_sizes[int(class_id)] = int((columns["connectivity_class_id"] == class_id).sum())

# relations within multi-member classes, checked against the panel's counts
relation_counts_by_position = {position: {"duplicate": 0, "enantiomer": 0, "diastereomer": 0, "stereo_undefined": 0} for position in range(len(molecules))}
classes_with_excluded_members = []
for class_id, positions in classes.items():
    if len(positions) < 2 and panel_class_sizes[class_id] == 1:
        continue
    for index_a, position_a in enumerate(positions):
        for position_b in positions[index_a + 1:]:
            relation = classify_pair(molecules[position_a], molecules[position_b])
            relation_counts_by_position[position_a][relation] += 1
            relation_counts_by_position[position_b][relation] += 1
    if panel_class_sizes[class_id] != len(positions):
        classes_with_excluded_members.append(class_id)
        continue
    for position in positions:
        row = molecules[position]["row"]
        recomputed = relation_counts_by_position[position]
        for relation, column_name in (("duplicate", "num_duplicates"), ("enantiomer", "num_enantiomers"), ("diastereomer", "num_diastereomers")):
            assert recomputed[relation] == int(columns[column_name][row]), f"row {row}: recomputed {relation} count {recomputed[relation]} vs panel {columns[column_name][row]}"
print(f"connectivity classes: {len(classes)}; multi-member classes: {sum(len(positions) > 1 for positions in classes.values())}")
print(f"classes that lost members to the valence-error exclusion (count gate skipped): {classes_with_excluded_members}")

# the split
for class_id, positions in classes.items():
    representative = min(positions, key=lambda position: molecules[position]["row"])
    for position in positions:
        molecule = molecules[position]
        molecule["representative_row"] = molecules[representative]["row"]
        if position == representative:
            molecule["role"] = "probe"
            molecule["relation_to_representative"] = None
        else:
            molecule["role"] = "gate"
            molecule["relation_to_representative"] = classify_pair(molecule, molecules[representative])
probe_positions = [position for position, molecule in enumerate(molecules) if molecule["role"] == "probe"]
gate_positions = [position for position, molecule in enumerate(molecules) if molecule["role"] == "gate"]
gate_relation_counts = {}
for position in gate_positions:
    relation = molecules[position]["relation_to_representative"]
    gate_relation_counts[relation] = gate_relation_counts.get(relation, 0) + 1
print(f"probe population: {len(probe_positions)}; gate set: {len(gate_positions)} {gate_relation_counts}")
assert len(probe_positions) + len(gate_positions) == len(molecules)

connectivity classes: 2118; multi-member classes: 18
classes that lost members to the valence-error exclusion (count gate skipped): []
probe population: 2118; gate set: 18 {'duplicate': 18}


## Folds: five, with every formula bucket kept whole

Buckets are ordered largest first, ties broken by a seed-0 permutation, and each bucket goes to the fold with the fewest molecules so far. Only the probe population is folded; gate molecules are never probe rows.

In [28]:
def assign_buckets_to_folds(bucket_sizes, num_folds, seed):
    """Greedy balanced assignment: largest bucket first, seeded tie-break, each bucket to the currently lightest fold"""
    bucket_keys = list(bucket_sizes.keys())
    tie_break = np.random.default_rng(seed).permutation(len(bucket_keys))
    order = sorted(range(len(bucket_keys)), key=lambda index: (-bucket_sizes[bucket_keys[index]], tie_break[index]))
    fold_load = [0] * num_folds
    bucket_to_fold = {}
    for index in order:
        lightest_fold = min(range(num_folds), key=lambda fold: (fold_load[fold], fold))
        bucket_to_fold[bucket_keys[index]] = lightest_fold
        fold_load[lightest_fold] += bucket_sizes[bucket_keys[index]]
    return bucket_to_fold, fold_load


# formula buckets on the probe population; formula tuple and formula_class_id must agree
formula_bucket_sizes = {}
formula_class_by_formula = {}
for position in probe_positions:
    molecule = molecules[position]
    formula_bucket_sizes[molecule["formula"]] = formula_bucket_sizes.get(molecule["formula"], 0) + 1
    class_id = int(columns["formula_class_id"][molecule["row"]])
    assert formula_class_by_formula.setdefault(molecule["formula"], class_id) == class_id, f"formula {molecule['formula']} carries two formula_class_ids"
assert len(set(formula_class_by_formula.values())) == len(formula_class_by_formula), "two formulas share a formula_class_id"

bucket_to_fold, fold_load = assign_buckets_to_folds(formula_bucket_sizes, NUM_FOLDS, SEED)
for position in probe_positions:
    molecules[position]["fold"] = bucket_to_fold[molecules[position]["formula"]]
for position in gate_positions:
    molecules[position]["fold"] = None

fold_sizes = [sum(1 for position in probe_positions if molecules[position]["fold"] == fold) for fold in range(NUM_FOLDS)]
buckets_per_fold = [sum(1 for fold_of_bucket in bucket_to_fold.values() if fold_of_bucket == fold) for fold in range(NUM_FOLDS)]
assert fold_sizes == fold_load and sum(fold_sizes) == len(probe_positions)
for formula, fold in bucket_to_fold.items():
    folds_seen = {molecules[position]["fold"] for position in probe_positions if molecules[position]["formula"] == formula}
    assert folds_seen == {fold}, f"formula {formula} appears in folds {folds_seen}"
print(f"formula buckets: {len(formula_bucket_sizes)}; largest {max(formula_bucket_sizes.values())}, singletons {sum(size == 1 for size in formula_bucket_sizes.values())}")
print(f"fold sizes: {fold_sizes}; buckets per fold: {buckets_per_fold}")
largest_buckets = sorted(formula_bucket_sizes.items(), key=lambda item: -item[1])[:10]
for formula, size in largest_buckets:
    formula_string = "".join(f"{ELEMENT_SYMBOLS[element]}{count}" for element, count in zip(ELEMENTS, formula) if count > 0)
    print(f"  {formula_string:12s} {size:5d} molecules -> fold {bucket_to_fold[formula]}")

formula buckets: 62; largest 184, singletons 3
fold sizes: [424, 424, 424, 423, 423]; buckets per fold: [12, 13, 13, 12, 12]
  H4C4N2O2       184 molecules -> fold 0
  H3C4N3O2       163 molecules -> fold 1
  H4C5N2O1       145 molecules -> fold 2
  H3C6N1O2       122 molecules -> fold 3
  H3C5N3O1       121 molecules -> fold 4
  H3C5N1O3       113 molecules -> fold 4
  H5C4N1O2       109 molecules -> fold 3
  H3C3N3O3        90 molecules -> fold 2
  H4C3N4O1        84 molecules -> fold 1
  H3C4N2O2F1      77 molecules -> fold 0


## Targets: the whole panel, with flags

Nothing is excluded. Every scalar panel field is a target, and the table reports for each one the distinct-value count, the off-mode support, the NaN count, and any fold whose test molecules show fewer than `MIN_DISTINCT_PER_TEST_FOLD` distinct values. These are flags that travel with the probe results, not filters. A constant target gets an undefined R² in the probe, reported as undefined. A target with NaNs gets a defined-subset mask so the probe fits on the molecules where it is defined. The probe treats pooled out-of-fold R² as primary and per-fold means as secondary, and never averages a flagged fold.

In [29]:
probe_rows = np.array([molecules[position]["row"] for position in probe_positions])
probe_folds = np.array([molecules[position]["fold"] for position in probe_positions])
target_matrix = np.column_stack([np.asarray(columns[name])[probe_rows].astype(float) for name in TARGET_NAMES])
assert target_matrix.shape == (len(probe_positions), len(TARGET_NAMES))

target_support = {}           # name -> (distinct values, off-mode support, NaN count), computed on the defined rows
target_defined_mask = {}      # name -> bool array over probe rows, False where the panel value is NaN
degenerate_folds = {}         # name -> folds whose defined test molecules show fewer than MIN_DISTINCT_PER_TEST_FOLD distinct values
constant_targets = []
for target_index, name in enumerate(TARGET_NAMES):
    values = target_matrix[:, target_index]
    defined = ~np.isnan(values)
    target_defined_mask[name] = defined
    distinct_values, value_counts = np.unique(values[defined], return_counts=True)
    off_mode_support = int(defined.sum() - value_counts.max())
    target_support[name] = (len(distinct_values), off_mode_support, int((~defined).sum()))
    if len(distinct_values) < 2:
        constant_targets.append(name)
    flagged = [fold for fold in range(NUM_FOLDS)
               if len(np.unique(values[defined & (probe_folds == fold)])) < MIN_DISTINCT_PER_TEST_FOLD]
    if flagged:
        degenerate_folds[name] = flagged

print(f"targets: {len(TARGET_NAMES)}, all carried; constant: {len(constant_targets)}; with NaNs: {sum(nan_count > 0 for _, _, nan_count in target_support.values())}; with flagged folds: {len(degenerate_folds)}")
print(f"{'target':45s} {'distinct':>8s} {'off-mode':>8s} {'NaN':>5s}  flags")
for block, names in TARGET_BLOCKS.items():
    print(f"[{block}]")
    for name in names:
        distinct, off_mode, nan_count = target_support[name]
        flags = []
        if name in constant_targets:
            flags.append("constant")
        if nan_count:
            flags.append("masked where NaN")
        if name in degenerate_folds:
            flags.append(f"folds {degenerate_folds[name]} degenerate")
        print(f"  {name:43s} {distinct:8d} {off_mode:8d} {nan_count:5d}  {', '.join(flags)}")

targets: 125, all carried; constant: 8; with NaNs: 2; with flagged folds: 13
target                                        distinct off-mode   NaN  flags
[composition]
  num_atoms                                          1        0     0  constant, folds [0, 1, 2, 3, 4] degenerate
  num_heavy_atoms                                    6     1056     0  
  num_bonds_full                                     5      636     0  
  num_bonds_heavy                                    9     1246     0  
  num_single_bonds_heavy                            11     1428     0  
  num_double_bonds_heavy                             5     1278     0  
  num_triple_bonds_heavy                             5      808     0  
  num_aromatic_bonds_heavy                           6     1311     0  
[rings]
  num_rings                                          5      636     0  
  num_aromatic_rings                                 3      952     0  
  num_nonaromatic_rings                              5      59

## Sector vocabularies

Each quotient sums Born probabilities over outcomes with the same occupancy signature: excited-atom counts per class. Element classes are H, C, N, O, F; degree classes are 1 to 4; joint classes are the (element, degree) pairs present anywhere in the census. The vocabulary is the union over molecules of every signature the molecule can produce, in a fixed lexicographic order, so every notebook downstream builds the same columns.

In [30]:
def occupancy_box(class_sizes):
    """Every occupancy tuple a molecule with these class sizes can produce: 0..n_c in each class"""
    return itertools.product(*[range(int(size) + 1) for size in class_sizes])


joint_classes = sorted({(int(element), int(degree)) for molecule in molecules for element, degree in zip(molecule["atomic_numbers"], molecule["degrees"])})
element_sectors = set()
degree_sectors = set()
joint_sectors = set()
for molecule in molecules:
    element_sizes = molecule["formula"]
    degree_sizes = [int((molecule["degrees"] == degree).sum()) for degree in DEGREE_CLASSES]
    joint_sizes = [int(((molecule["atomic_numbers"] == element) & (molecule["degrees"] == degree)).sum()) for element, degree in joint_classes]
    assert sum(element_sizes) == sum(degree_sizes) == sum(joint_sizes) == CENSUS_NUM_ATOMS
    element_sectors.update(occupancy_box(element_sizes))
    degree_sectors.update(occupancy_box(degree_sizes))
    joint_sectors.update(occupancy_box(joint_sizes))
element_sectors = sorted(element_sectors)
degree_sectors = sorted(degree_sectors)
joint_sectors = sorted(joint_sectors)
print(f"joint classes: {[(ELEMENT_SYMBOLS[element], degree) for element, degree in joint_classes]}")
print(f"vocabulary sizes: element {len(element_sectors)}, degree {len(degree_sectors)}, joint {len(joint_sectors)}")

joint classes: [('H', 1), ('C', 2), ('C', 3), ('C', 4), ('N', 1), ('N', 2), ('N', 3), ('O', 1), ('O', 2), ('F', 1)]
vocabulary sizes: element 1556, degree 830, joint 20116


## Digests and save

In [31]:
def digest_of(text):
    """First 16 hex digits of sha256"""
    return hashlib.sha256(text.encode()).hexdigest()[:16]


for molecule in molecules:
    molecule["graph_digest"] = digest_of(f"{molecule['atomic_numbers'].tolist()}|{molecule['edges'].tolist()}|{molecule['bond_types']}")
    molecule["geometry_digest"] = digest_of(",".join(f"{length:.9f}" for length in molecule["bond_lengths"]))
probe_population_digest = digest_of(",".join(str(molecules[position]["qm9_index"]) for position in probe_positions))
gate_set_digest = digest_of(",".join(str(molecules[position]["qm9_index"]) for position in gate_positions))
fold_digest = digest_of(",".join(f"{molecules[position]['qm9_index']}:{molecules[position]['fold']}" for position in probe_positions))
target_digest = digest_of(",".join(TARGET_NAMES) + "|" + hashlib.sha256(target_matrix.tobytes()).hexdigest())

config = {
    "PANEL_PKL": PANEL_PKL, "QM9_ROOT": QM9_ROOT, "OUTPUT_PKL": OUTPUT_PKL,
    "CENSUS_NUM_ATOMS": CENSUS_NUM_ATOMS, "EXPECTED_MAX_DEGREE": EXPECTED_MAX_DEGREE,
    "NUM_FOLDS": NUM_FOLDS, "SEED": SEED,
    "MIN_DISTINCT_PER_TEST_FOLD": MIN_DISTINCT_PER_TEST_FOLD,
    "ELEMENTS": ELEMENTS, "PYG_BOND_TYPES": PYG_BOND_TYPES, "DEGREE_CLASSES": DEGREE_CLASSES,
    "TARGET_BLOCKS": TARGET_BLOCKS, "NON_TARGET_FIELDS": NON_TARGET_FIELDS,
    "panel_num_molecules": num_panel_molecules,
}
freeze = {
    "config": config,
    "census_constants": {
        "num_atoms": CENSUS_NUM_ATOMS,
        "max_degree": census_max_degree,
        "longest_bond_length": census_longest_bond,
        "shortest_bond_length": census_shortest_bond,
        "valence_error_excluded": int((at_census_size & valence_error).sum()),
    },
    "molecules": molecules,                        # both roles; the producer runs every one of them
    "probe_rows": probe_rows,                      # panel rows of the probe population, in order
    "probe_qm9_indices": np.array([molecules[position]["qm9_index"] for position in probe_positions]),
    "gate_rows": np.array([molecules[position]["row"] for position in gate_positions]),
    "folds": {"probe_fold": probe_folds, "bucket_to_fold": bucket_to_fold, "fold_sizes": fold_sizes, "buckets_per_fold": buckets_per_fold},
    "targets": {"names": TARGET_NAMES, "blocks": TARGET_BLOCKS, "matrix": target_matrix,     # NaN kept where the panel has NaN
                "defined_mask": target_defined_mask, "support": target_support,
                "constant": constant_targets, "degenerate_folds": degenerate_folds},
    "vocabularies": {"element_classes": ELEMENTS, "element_sectors": element_sectors,
                     "degree_classes": DEGREE_CLASSES, "degree_sectors": degree_sectors,
                     "joint_classes": joint_classes, "joint_sectors": joint_sectors},
    "digests": {"probe_population": probe_population_digest, "gate_set": gate_set_digest, "folds": fold_digest, "targets": target_digest},
}
with open(OUTPUT_PKL, "wb") as handle:
    pickle.dump(freeze, handle)
print(f"saved {OUTPUT_PKL}")
print(f"probe population {len(probe_positions)} (digest {probe_population_digest}), gate set {len(gate_positions)} (digest {gate_set_digest})")
print(f"folds digest {fold_digest}; targets digest {target_digest}")

saved iclr26_freeze_n12.pkl
probe population 2118 (digest ccac21c24c2ed225), gate set 18 (digest eff321d298799891)
folds digest 891302c4d0902607; targets digest 70c24fbf6f7cf1be


## What the producer reads

Each entry of `freeze["molecules"]` carries `atomic_numbers`, `degrees`, `edges` (sorted, undirected), `bond_types` (RDKit-perceived, the panel's convention, used by the bond-type arm), `kekule_bond_types` (PyG's SDF types, provenance only), and `bond_lengths`, plus `role` (`probe` or `gate`), `fold` (probe only), `representative_row`, and `relation_to_representative` (gate only: duplicate, enantiomer, diastereomer, or stereo_undefined). `freeze["census_constants"]` gives the maximum degree and the longest bond for the degree and bond-length normalizations. The producer runs both roles; the probe uses `probe_rows`, `folds`, and `targets`; the audit uses the gate set.

Notes carried forward:
- Every panel field is a target. `girth` keeps the panel's convention, 0 for acyclic molecules; the probe may restrict it to the cyclic subset, and `cyclomatic_number` is the mask. Targets with NaNs carry `defined_mask`; constant targets are listed in `constant` and get an undefined R².
- Test folds contain formulas the probe never saw in training. Pooled out-of-fold R² is the primary statistic; per-fold means are secondary and skip flagged folds.